In [1]:
source ../modules/setup.tcl

set year 2021
set day 9

aoc::get-puzzle $year $day 1
aoc::get-puzzle $year $day 2
set input [string trim [aoc::get-input $year $day]]
jupyter::display "text/markdown" "## Input\n```\n[string range $input 0 20]...\n```";

(cached)

--- Day 9: Smoke Basin --- These caves seem to be lava tubes . Parts are even still volcanically active; small hydrothermal vents release smoke into the caves that slowly settles like rain . 
 If you can model how the smoke flows through the caves, you might be able to avoid it and be that much safer. The submarine generates a heightmap of the floor of the nearby caves for you (your puzzle input). 
 Smoke flows to the lowest point of the area it's in. For example, consider the following heightmap: 
 2 1 9994321 0 
3987894921
98 5 6789892
8767896789
989996 5 678
 
 Each number corresponds to the height of a particular location, where 9 is the highest and 0 is the lowest a location can be. 
 Your first goal is to find the low points - the locations that are lower than any of its adjacent locations. Most locations have four adjacent locations (up, down, left, and right); locations on the edge or corner of the map have three or two adjacent locations, respectively. (Diagonal locations do not count as adjacent.) 
 In the above example, there are four low points, all highlighted: two are in the first row (a 1 and a 0 ), one is in the third row (a 5 ), and one is in the bottom row (also a 5 ). All other locations on the heightmap have some lower adjacent location, and so are not low points. 
 The risk level of a low point is 1 plus its height . In the above example, the risk levels of the low points are 2 , 1 , 6 , and 6 . The sum of the risk levels of all low points in the heightmap is therefore 15 . 
 Find all of the low points on your heightmap. What is the sum of the risk levels of all low points on your heightmap?

(cached)

--- Part Two --- Next, you need to find the largest basins so you know what areas are most important to avoid. 
 A basin is all locations that eventually flow downward to a single low point. Therefore, every low point has a basin, although some basins are very small. Locations of height 9 do not count as being in any basin, and all other locations will always be part of exactly one basin. 
 The size of a basin is the number of locations within the basin, including the low point. The example above has four basins. 
 The top-left basin, size 3 : 
 
 21 99943210
 3 987894921
9856789892
8767896789
9899965678
 
 The top-right basin, size 9 : 
 21999 43210 
398789 4 9 21 
985678989 2 
8767896789
9899965678
 
 The middle basin, size 14 : 
 2199943210
39 878 94921
9 85678 9892
 87678 96789
9 8 99965678
 
 The bottom-right basin, size 9 : 
 2199943210
3987894921
9856789 8 92
876789 678 9
98999 65678 
 
 Find the three largest basins and multiply their sizes together. In the above example, this is 9 * 14 * 9 = 1134 
 . 
 What do you get if you multiply together the sizes of the three largest basins?

(cached)

## Input
```
896979998764679896543...
```

### Solve today

`aoc::solve input body`:
    The body is executed as a coroutine. Input is available as the `$input` variable. The results are returned using `[yield]`

In [2]:
set test {2199943210
3987894921
9856789892
8767896789
9899965678}

package require struct::graph
package require struct::graph::op


aoc::solve $input {
    set input [aoc::togrid $input]
    set dy [llength $input]
    set dx [llength [lindex $input 0]]
    set lps {}
    struct::graph mygraph
    for {set y 0} {$y < $dy} {incr y} {
        set row [lindex $input $y]
        for {set x 0} {$x < $dx} {incr x} {
            set pval [lindex $input $y $x]
            set lp true
            if {![mygraph node exists [list $x $y]]} {
                mygraph node insert [list $x $y]
            }
            foreach np [aoc::neighbours4 $x $y] {
                lassign $np nx ny
                if {![mygraph node exists $np]} {
                    mygraph node insert $np
                }
                
                if {$nx < 0 || $nx >= $dx || $ny < 0 || $ny >= $dy} continue
                set npval [lindex $input $ny $nx]
                if {$npval != 9 && $pval != 9} {
                    mygraph arc insert [list $x $y] $np
                }
                if {$npval <= $pval} {
                    set lp false
                }
            }
            if {$lp} {
                lappend lps [list [list $x $y] $pval]
            }
        }
    }
    yield [+ {*}[lmap lp $lps {+ [lindex $lp end] 1}]]
    set result2  [lrange [lsort -index 0 -decreasing -integer [lmap s [struct::graph::op::connectedComponents mygraph] {llength $s}]] 0 2]
    mygraph destroy
    yield [* {*}$result2]
}


Part1	522 (543054 us)
Part2	916688 (1650463 us)
